# 1. Preparação e Auditoria dos Microdados — PNAD Covid-19 (2020)

Este projeto investiga os determinantes do trabalho remoto no Brasil durante a pandemia da COVID-19 (maio a novembro de 2020), com foco central no papel do **ensino superior** e em suas **interações com características individuais e ocupacionais**.

Neste primeiro notebook realizamos:
1. A carga e auditoria da base compacta da **PNAD Covid-19 (IBGE)** contendo as 18 variáveis essenciais das 7 edições mensais (2.650.459 observações);
2. A verificação empírica das **regras de salto do questionário do IBGE** (`C001 → C012 → C013` e `C007 → C007B / C014 / C007D`) para definir com precisão o universo ocupado e a variável dependente;
3. A engenharia vetorizada de variáveis e a validação de integridade antes de gravar a base analítica tratada em `data/processed/pnad_covid_analitico.parquet`.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Garante resolução da raiz do projeto independentemente do diretório de execução
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_DICTIONARY, MES_LABELS, PROCESSED_DATA_PATH, RAW_DATA_PATH
from src.data_processing import (
    build_analytic_dataset,
    load_raw_data,
    save_analytic_dataset,
    validate_analytic_dataset,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Base bruta:      {RAW_DATA_PATH}")


Raiz do projeto: C:\Codigos\COVID-Trabalho-Remoto
Base bruta:      C:\Codigos\COVID-Trabalho-Remoto\data\raw\pnad_covid_2020.parquet


## 1.1 Carga e Auditoria Inicial da Base Bruta

A base de entrada `data/raw/pnad_covid_2020.parquet` consolida as edições de maio (`V1013 = 5`) a novembro (`V1013 = 11`) de 2020, mantendo apenas as colunas demográficas, educacionais, ocupacionais e o peso amostral calibrado (`V1032`).


In [2]:
df_raw = load_raw_data()
mem_mb = df_raw.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Dimensões da base bruta: {df_raw.shape[0]:,} linhas x {df_raw.shape[1]} colunas")
print(f"Uso aproximado em RAM:   {mem_mb:.1f} MB")

auditoria_mes = (
    df_raw.assign(mes_label=df_raw["V1013"].map(MES_LABELS))
    .groupby(["V1013", "mes_label"], observed=True)
    .agg(
        entrevistas=("V1032", "size"),
        populacao_estimada_mi=("V1032", lambda s: s.sum() / 1e6),
        peso_medio=("V1032", "mean"),
    )
    .round(2)
)
display(auditoria_mes)


Dimensões da base bruta: 2,650,459 linhas x 18 colunas
Uso aproximado em RAM:   68.3 MB


,,entrevistas,populacao_estimada_mi,peso_medio
V1013,mes_label,,,
5,mai/2020,349306,210.87,603.68
6,jun/2020,381270,211.00,553.42
7,jul/2020,384166,211.13,549.58
8,ago/2020,386520,211.26,546.57
9,set/2020,387298,211.39,545.81
10,out/2020,380461,211.52,555.96
11,nov/2020,381438,211.65,554.88


## 1.2 Regras de Salto do Questionário e Universo Elegível

Na PNAD Covid-19, o bloco de trabalho (Módulo C) possui filtros encadeados fundamentais para evitar viés de seleção na modelagem:

- **`C001` (*Trabalhou na semana de referência?*)**: responde à ocupação efetiva (não afastada).
- **`C012` (*Exerceu o trabalho no mesmo local em que costumava trabalhar?*)**: perguntada apenas a quem respondeu `C001 == 'Sim'` e exerce atividade no mercado de trabalho (pessoas fora do mercado que faziam apenas afazeres domésticos ou produção para consumo próprio ficam com `C012 = NaN`).
- **`C013` (*Estava em trabalho remoto / home office?*)**: aplicada **exclusivamente** a quem respondeu `C012 == 'Não'`. Logo, trabalhadores que permaneceram em seu local habitual (`C012 == 'Sim'`) têm `C013 = NaN` por salto estrutural — são trabalhadores presenciais (`trabalho_remoto = 0`), e **não** observações perdidas.

Além disso:
- **Militares (`C007` policial militar, bombeiro ou forças armadas)**: possuem `C007B`, `C007D` e `C014` nulos no questionário e são classificados explicitamente em `iniciativa = 'Público'`, `formalidade = 'Formal'` e `setor = 'Setor Público'`.
- **Formalidade (`C007`, `C007B`, `C014`)**: incorpora trabalhadores com carteira assinada, estatutários, militares e trabalhadores por conta própria / empregadores / sem carteira que contribuem para a Previdência Social (`C014 == 'Sim'`), alinhando-se ao conceito padrão do IBGE.

A tabela abaixo comprova empiricamente o padrão exato dos saltos `C001 × C012 × C013` nos microdados.


In [3]:
saltos = (
    df_raw.groupby(["C001", "C012", "C013"], dropna=False, observed=False)
    .agg(
        registros=("V1032", "size"),
        pop_milhoes=("V1032", lambda s: round(s.sum() / 1e6, 2)),
    )
    .query("registros > 0")
)
display(saltos)

# Confirmação sobre C007D nulo entre elegíveis (C001=='Sim' & C012 não nulo)
elig_c007d_nulo = df_raw.loc[
    (df_raw["C001"] == "Sim") & df_raw["C012"].notna() & df_raw["C007D"].isna(),
    "C007",
].value_counts()
print("\nRegistros elegíveis com C007D nulo (apenas militares):")
display(elig_c007d_nulo.to_frame("registros"))


registros  pop_milhoes
C001 C012 C013                        
Sim  Sim  nan      774875       442.12
     Não  Sim       83799        57.18
          Não       34571        19.28
     nan  nan       32417        12.34
Não  nan  nan     1259629       661.52
nan  nan  nan      465168       286.40


Registros elegíveis com C007D nulo (apenas militares):


,registros
C007,
Policial militar ou bombeiro militar,4297
"Militar do exercito, marinha ou aeronáutica",3824
"Trabalhador doméstico (empregado doméstico, cuidados, babá)",0
Empregado do setor privado,0
Empregado do setor público (inclusive empresas de economia mista),0
Empregador,0
Conta própria,0
Trabalhador familiar não remunerado em ajuda a membro do domicílio ou parente,0
"Estava fora do mercado de trabalho (fazia apenas afazeres domésticos, cuidados de pessoas ou produção para próprio consumo)",0


## 1.3 Construção da Base Analítica e Dicionário de Variáveis

Aplicamos `build_analytic_dataset(df_raw)` para gerar todas as variáveis categóricas ordenadas, indicadores binários (`ensino_superior`, `trabalho_remoto`) e a flag `amostra_modelo`.


In [4]:
df_analytic = build_analytic_dataset(df_raw)
print(f"Base analítica construída: {df_analytic.shape[0]:,} linhas x {df_analytic.shape[1]} colunas")

dicionario_df = pd.DataFrame(DATA_DICTIONARY)
display(dicionario_df)


Base analítica construída: 2,650,459 linhas x 24 colunas


,variavel,origem,tipo,descricao
0,ano,Ano (IBGE),int16,Ano de referência da pesquisa (2020).
1,mes,V1013 (IBGE),int8,Mês da pesquisa (5 = maio a 11 = novembro de 2020).
2,mes_label,Derivada (V1013),category,Rótulo cronológico mês/ano (mai/2020 a nov/2020).
3,semana,V1012 (IBGE),int8,Semana no mês (1 a 4).
4,uf,UF (IBGE),category,Unidade da Federação de residência.
5,uf_sigla,Derivada (UF),category,Sigla oficial da UF (27 UFs).
6,regiao,Derivada (UF),category,"Macrorregião geográfica (Sudeste, Nordeste, Sul, Centro-Oeste, Norte)."
7,peso,V1032 (IBGE),float64,Peso amostral calibrado para expansão populacional.
8,idade,A002 (IBGE),int16,Idade do morador em anos.
9,faixa_etaria_desc,Derivada (A002),category,Faixa etária descritiva (14-19 até 80+).


## 1.4 Validações de Consistência e Persistência em Parquet

Validamos que:
1. A amostra do modelo (`amostra_modelo == True`) possui **zero valores ausentes** em todas as covariáveis e na variável resposta;
2. Nenhum indivíduo fora dos ocupados não afastados aparece como `trabalho_remoto == 1`;
3. Os totais mensais ponderados reproduzem fielmente os contingentes nacionais da pesquisa (pico de 8,71 milhões em trabalho remoto em maio/2020 e 7,33 milhões em novembro/2020).


In [5]:
checks = validate_analytic_dataset(df_analytic)
assert all(v == 0 for v in checks["nulos_amostra_modelo"].values()), "Nulos inesperados na amostra_modelo!"
assert checks["remoto_fora_nao_afastado"] == 0, "Remoto fora de ocupados não afastados!"

print(f"Total de registros:              {checks['n_total_linhas']:,}")
print(f"Registros elegíveis para Logit:  {checks['n_amostra_modelo']:,}")
print(f"UFs cobertas:                    {checks['n_ufs']}")
print(f"Nulos na amostra_modelo:         {checks['nulos_amostra_modelo']}")
print("\nResumo mensal ponderado (milhões de pessoas) e tamanho da amostra do modelo:")
display(checks["resumo_mensal"])

saved_path = save_analytic_dataset(df_analytic, PROCESSED_DATA_PATH)
size_mb = saved_path.stat().st_size / (1024 ** 2)
print(f"\nBase analítica salva com sucesso em: {saved_path} ({size_mb:.2f} MB)")


Total de registros:              2,650,459
Registros elegíveis para Logit:  893,086
UFs cobertas:                    27
Nulos na amostra_modelo:         {'trabalho_remoto': 0, 'ensino_superior': 0, 'escolaridade': 0, 'genero': 0, 'raca': 0, 'faixa_etaria_modelo': 0, 'regiao': 0, 'iniciativa': 0, 'formalidade': 0, 'setor': 0, 'peso': 0}

Resumo mensal ponderado (milhões de pessoas) e tamanho da amostra do modelo:


,ocupados_milhoes,nao_afastados_milhoes,remotos_milhoes,amostra_modelo_n
mes,,,,
5,86.744,66.949,8.709,106686.0
6,86.444,70.298,8.694,120305.0
7,86.860,73.427,8.403,125235.0
8,87.637,77.277,8.376,131835.0
9,87.983,79.439,8.073,135383.0
10,88.686,81.377,7.596,136021.0
11,88.949,82.147,7.330,137621.0



Base analítica salva com sucesso em: C:\Codigos\COVID-Trabalho-Remoto\data\processed\pnad_covid_analitico.parquet (21.86 MB)
